# Location ID -> Country name

Build a country-name -> location_id JSON lookup table from country_ids.csv
(the exported 'data' tab of location_hierarchy_s1_r7_v1524_20260217_150124.xlsx).  

Produces:
    country_location_ids.json  -- {"Armenia": 33, "Azerbaijan": 34, ...}

In [3]:
import os
import json
import xarray as xr
import pandas as pd
import config
from utils.utils import require_dir
import pathlib

In [4]:
# === Path config ===
DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "GBD23")
MASKS_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR" / "masks" / "country", "country masks")

In [5]:
def build_lookup(csv_path, countries, overrides):
    df = pd.read_csv(csv_path)
    by_name = {name: g for name, g in df.groupby('location_name')}

    lookup = {}
    for country in countries:
        sheet_name = overrides.get(country, country)
        rows = by_name.get(sheet_name)
        if rows is None:
            raise KeyError(f"No match in sheet for {country!r} (looked up as {sheet_name!r})")
        if len(rows) == 1:
            row = rows.iloc[0]
        else:
            # Disambiguate duplicate names (e.g. "Georgia" country vs US state)
            # by preferring the country-level (admin0) row.
            admin0 = rows[rows['location_type'] == 'admin0']
            if len(admin0) != 1:
                raise ValueError(f"Ambiguous match for {country!r}: {len(rows)} candidate rows")
            row = admin0.iloc[0]
        lookup[country] = int(row['location_id'])

    return lookup

23 of the names above don't appear verbatim in the sheet, because the sheet uses official/UN-style names for these entities.  
Map input name -> sheet location_name for those cases only.

In [6]:
overrides = {
    'Czech Republic': 'Czechia',
    'Macedonia': 'North Macedonia',
    'Moldova': 'Republic of Moldova',
    'Brunei': 'Brunei Darussalam',
    'South Korea': 'Republic of Korea',
    'United States': 'United States of America',
    'Bolivia': 'Bolivia (Plurinational State of)',
    'The Bahamas': 'Bahamas',
    'Virgin Islands, U.S.': 'United States Virgin Islands',
    'Venezuela': 'Venezuela (Bolivarian Republic of)',
    'Iran': 'Iran (Islamic Republic of)',
    'Syria': 'Syrian Arab Republic',
    'Turkey': 'Türkiye',
    'North Korea': "Democratic People's Republic of Korea",
    'Taiwan': 'Taiwan (Province of China)',
    'Federated States of Micronesia': 'Micronesia (Federated States of)',
    'Laos': "Lao People's Democratic Republic",
    'Vietnam': 'Viet Nam',
    'Tanzania': 'United Republic of Tanzania',
    'Swaziland': 'Eswatini',
    'Cape Verde': 'Cabo Verde',
    "Cote d'Ivoire": "Côte d'Ivoire",
    'The Gambia': 'Gambia',
}

In [7]:
# set up country ids
csv_file = "country_ids.csv"
csv_path = os.path.join(DIR, csv_file)

# Load country masks
mask_file = "GBD_Country_Masks_0.10.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
masks = xr.open_dataarray(mask_path)

In [8]:
countries = masks.country.values

lookup = build_lookup(csv_path, countries, overrides)
print(f"Matched {len(lookup)} of {len(countries)} countries")

out_file = "country_location_ids.json"
out_path = os.path.join(DIR, out_file)

with open(out_path, "w", encoding="utf-8") as f:
    print(f"Saving to {out_path}")
    json.dump(lookup, f, indent=2, ensure_ascii=False)

print("All processing complete.")

Matched 204 of 204 countries
Saving to /glade/work/awells/workflow/GBD23/country_location_ids.json
All processing complete.
